# 02 - LangChain + OCI Generative AI using `langchain-oci`

This notebook follows the written guide: `../LangChainOCI-GenAIOCI-Connection.md`.

The goal is to show how LangChain can call OCI Generative AI with less SDK plumbing:

1. Create a LangChain chat model with `ChatOCIGenAI`.
2. Invoke the model with simple chat messages.
3. Build a prompt chain with LCEL, the LangChain Expression Language.
4. Create OCI embeddings with `OCIGenAIEmbeddings`.
5. Use a tiny in-memory retrieval example to connect the concepts.

Official references:

- OCI Generative AI LangChain integration: https://docs.oracle.com/en-us/iaas/Content/generative-ai/langchain.htm
- LangChain OCI package: https://python.langchain.com/docs/integrations/providers/oci/
- LangChain prompt templates and LCEL: https://python.langchain.com/docs/concepts/lcel/


Copyright (c) 2026 Oracle and/or its affiliates. Licensed under the Universal Permissive License (UPL), Version 1.0.


## How This Relates To The Written Guide

The written guide introduces three classes:

| Class | Purpose |
|---|---|
| `ChatOCIGenAI` | Chat models on OCI Generative AI |
| `OCIGenAI` | Completion-style LLMs on OCI Generative AI |
| `OCIGenAIEmbeddings` | Embedding models on OCI Generative AI |

Because the current `.env` setup uses Gemini as the chat model, this notebook uses `ChatOCIGenAI`. The written guide's `OCIGenAI` examples are still useful for completion-style models, but `ChatOCIGenAI` is the better match for this configuration.

The written guide also shows `LLMChain`. Current LangChain versions use LCEL chains instead, for example:

```python
chain = prompt | chat_model | output_parser
```


## What LangChain Is

LangChain is a Python framework for building applications around large language models. It gives you reusable building blocks for prompts, chat models, output parsing, retrieval, and agents.

A simple way to think about it:

```text
your inputs -> prompt template -> model -> cleaned output
```

Common LangChain patterns are:

| Pattern | Plain meaning | Example in this notebook |
|---|---|---|
| Chat model | A standard way to call a chat model | `ChatOCIGenAI` |
| Prompt template | A reusable prompt with variables | `ChatPromptTemplate` |
| Chain | Connect steps together | `prompt | chat_model | StrOutputParser()` |
| Output parser | Convert the model response into the shape you want | `StrOutputParser()` |
| Embeddings | Turn text into vectors for search | `OCIGenAIEmbeddings` |
| Retrieval / RAG | Find relevant context, then ask the model to answer from it | Steps 7 and 8 |

OCI Generative AI integrates directly through the `langchain-oci` package. That package gives LangChain classes for OCI chat models, completion models, and embedding models. In practice, you still use the same OCI values from notebook `00`: model ID, service endpoint, compartment ID, and API-key authentication profile.

Important point for this demo: LangChain is not replacing OCI Generative AI. LangChain is the application framework. OCI Generative AI is still the service doing the chat and embedding work.

Official docs behind this section:

- LangChain overview: https://docs.langchain.com/oss/python/langchain/overview
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- Oracle OCI Generative AI LangChain integration: https://docs.oracle.com/en-us/iaas/Content/generative-ai/langchain.htm
- LangChain OCI provider docs: https://docs.langchain.com/oss/python/integrations/providers/oci


## Install Python Packages

Install dependencies once from the `files` directory by running `uv sync` before opening this notebook.


In [ ]:
# Dependencies are managed by uv. Run `uv sync` from the files directory before opening this notebook.

## Step 1 - Load Workshop Setup

This reuses the same `.env`, OCI config profile, endpoint, compartment, and model IDs from notebook `00`.


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import os
import sys

import pandas as pd

# Make the repo-level helper package importable from this notebook.
for directory in [Path.cwd(), *Path.cwd().parents]:
    if (directory / "workshop_helpers").exists():
        sys.path.insert(0, str(directory))
        break
else:
    raise FileNotFoundError("Could not find the workshop_helpers folder.")

from workshop_helpers.oci_genai_helpers import load_workshop_env, mask

# Load the same configuration used by the SDK notebook.
env_path = load_workshop_env()

display(Markdown(f"""
**Workshop Setup Loaded**

| Setting | Value |
|---|---|
| `.env` file | `{env_path}` |
| OCI profile | `{os.getenv('OCI_PROFILE', 'DEFAULT')}` |
| GenAI endpoint | `{os.getenv('GENAI_ENDPOINT')}` |
| Compartment | `{mask(os.getenv('OCI_COMPARTMENT_ID'), 32)}` |
| Chat model | `{os.getenv('CHAT_MODEL_ID')}` |
| Embedding model | `{os.getenv('EMBED_MODEL_ID')}` |
"""))

## Step 2 - Create The LangChain Chat Model

This is the LangChain equivalent of creating an OCI SDK `GenerativeAiInferenceClient` plus a chat request.

The constructor shape is intentionally shown here because these are the main values users need to recognize:

- `model_id`: the OCI model name, such as `google.gemini-2.5-flash`
- `service_endpoint`: the regional OCI Generative AI endpoint
- `compartment_id`: the OCI compartment OCID
- `auth_type`, `auth_profile`, `auth_file_location`: config-file authentication settings
- `model_kwargs`: generation parameters passed to the model


In [ ]:
from langchain_oci import ChatOCIGenAI

from workshop_helpers.langchain_oci_helpers import infer_provider

# These values come from .env so the notebook stays portable across tenancies.
chat_model_id = os.environ["CHAT_MODEL_ID"]
chat_provider = infer_provider(chat_model_id)
chat_max_tokens = int(os.getenv("CHAT_MAX_TOKENS", "1024"))
chat_max_completion_tokens = int(os.getenv("CHAT_MAX_COMPLETION_TOKENS", "8192"))

# Some reasoning-capable models need a larger output budget before they return visible text.
langchain_max_tokens = max(chat_max_tokens, chat_max_completion_tokens)

# This is the core langchain-oci chat model construction.
chat_model = ChatOCIGenAI(
    model_id=chat_model_id,
    provider=chat_provider,
    service_endpoint=os.environ["GENAI_ENDPOINT"],
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    auth_type="API_KEY",
    auth_profile=os.getenv("OCI_PROFILE", "DEFAULT"),
    auth_file_location=os.path.expanduser(os.getenv("OCI_CONFIG_FILE", "~/.oci/config")),
    model_kwargs={
        "temperature": 0.2,
        "top_p": 0.75,
        "max_tokens": langchain_max_tokens,
        "max_completion_tokens": chat_max_completion_tokens,
        "reasoning_effort": os.getenv("CHAT_REASONING_EFFORT", "LOW"),
    },
)

display(Markdown(f"""
**Chat Model Ready**

| Setting | Value |
|---|---|
| LangChain class | `ChatOCIGenAI` |
| OCI model | `{chat_model_id}` |
| Provider | `{chat_provider}` |
| Auth type | `API_KEY` |
| Auth profile | `{os.getenv('OCI_PROFILE', 'DEFAULT')}` |
| Max tokens | `{langchain_max_tokens}` |
"""))

## Step 3 - First Chat Invocation

LangChain represents a chat request as messages. This is useful because real chat applications usually have system instructions, user turns, and assistant turns.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Send one direct chat request through LangChain.
messages = [
    SystemMessage(content="You answer clearly in one short paragraph."),
    HumanMessage(content="What is OCI Generative AI?"),
]

response = chat_model.invoke(messages)

display(Markdown(f"""
**Model Response**

{response.content}

**Response Metadata**

| Field | Value |
|---|---|
| Finish reason | `{response.response_metadata.get('finish_reason')}` |
| Total tokens | `{response.usage_metadata.get('total_tokens') if response.usage_metadata else 'n/a'}` |
"""))

## Step 4 - Build A Prompt Chain With LCEL

The written guide shows prompt chaining. In current LangChain, the clearest way to do that is LCEL:

```python
chain = prompt | chat_model | output_parser
```

Each piece has one job: format the prompt, call the model, then return plain text.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def empty_model_output_note(context: str) -> str:
    return (
        f"_No visible text was returned for {context}. If you are using a reasoning-capable "
        "model such as Gemini, increase `CHAT_MAX_COMPLETION_TOKENS` in `.env` and rerun "
        "Step 2 so the LangChain model picks up the larger token budget._"
    )


def invoke_text_chain(chain, inputs: dict, context: str) -> str:
    # Keep notebook execution moving, but make empty model output visible.
    text = chain.invoke(inputs).strip()
    return text or empty_model_output_note(context)


# A prompt template lets us reuse one prompt with different inputs.
prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain technical topics for a {audience} audience."),
    ("human", "Explain {topic} in three concise bullets."),
])

# LCEL connects prompt -> model -> text output.
chain = prompt | chat_model | StrOutputParser()

chain_inputs = {
    "audience": "a mixed business and technical audience",
    "topic": "why LangChain is useful with OCI Generative AI",
}

answer = invoke_text_chain(chain, chain_inputs, "the Step 4 prompt chain")

display(Markdown(f"""
**Prompt Chain Inputs**

| Input | Value |
|---|---|
| Audience | {chain_inputs['audience']} |
| Topic | {chain_inputs['topic']} |

**Generated Answer**

{answer}
"""))

## Step 5 - Make The Chain Reusable

This is a common application pattern: wrap the chain in a small function, then call it with different inputs.


In [ ]:
def explain_for_audience(topic: str, audience: str) -> str:
    # The chain stays the same; only the input values change.
    return invoke_text_chain(
        chain,
        {"topic": topic, "audience": audience},
        f"the reusable chain for {audience}",
    )

business_answer = explain_for_audience(
    topic="Retrieval-Augmented Generation",
    audience="non-technical sales leaders",
)

technical_answer = explain_for_audience(
    topic="Retrieval-Augmented Generation",
    audience="Python developers",
)

display(Markdown(f"""
**For Non-Technical Sales Leaders**

{business_answer}

**For Python Developers**

{technical_answer}
"""))

## Step 6 - Create OCI Embeddings With LangChain

Embeddings turn text into vectors. LangChain gives us a standard interface, while OCI Generative AI still performs the actual embedding work.

The constructor mirrors the written guide's embedding setup: model ID, service endpoint, compartment ID, and OCI config-file authentication.


In [ ]:
from langchain_oci import OCIGenAIEmbeddings

# SEARCH_DOCUMENT is the right input type when embedding stored content.
document_embeddings = OCIGenAIEmbeddings(
    model_id=os.environ["EMBED_MODEL_ID"],
    service_endpoint=os.environ["GENAI_ENDPOINT"],
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    auth_type="API_KEY",
    auth_profile=os.getenv("OCI_PROFILE", "DEFAULT"),
    auth_file_location=os.path.expanduser(os.getenv("OCI_CONFIG_FILE", "~/.oci/config")),
    input_type="SEARCH_DOCUMENT",
)

sample_text = "OCI Generative AI can create embeddings for semantic search and RAG."
vector = document_embeddings.embed_documents([sample_text])[0]

display(Markdown(f"""
**Embedding Created**

| Field | Value |
|---|---|
| LangChain class | `OCIGenAIEmbeddings` |
| Text | {sample_text} |
| Vector dimensions | {len(vector)} |
| Embedding model | `{os.getenv('EMBED_MODEL_ID')}` |
| Input type | `SEARCH_DOCUMENT` |
"""))

## Step 7 - Tiny In-Memory Retrieval Example

This mirrors the RAG idea from notebook `01`, but without Oracle Database. LangChain gives us a consistent model interface; a production app would normally use a vector database or Oracle Database vector search for retrieval.


In [ ]:
from workshop_helpers.langchain_oci_helpers import top_matches

# Small document set for an easy-to-read retrieval example.
documents = [
    {
        "source": "oci-sdk-guide",
        "text": "The OCI SDK gives direct control over Generative AI requests, model parameters, and service clients.",
    },
    {
        "source": "langchain-guide",
        "text": "LangChain helps compose prompts, chat models, output parsers, retrievers, and application logic into reusable chains.",
    },
    {
        "source": "n8n-guide",
        "text": "n8n can use an OpenAI-compatible gateway to call OCI Generative AI from low-code workflows.",
    },
]

# Embed stored documents as searchable content.
document_texts = [doc["text"] for doc in documents]
document_vectors = document_embeddings.embed_documents(document_texts)

# Embed the user question as a search query. This uses the same embedding model with a different input type.
query = "How does LangChain make an OCI GenAI application easier to build?"
query_embeddings = OCIGenAIEmbeddings(
    model_id=os.environ["EMBED_MODEL_ID"],
    service_endpoint=os.environ["GENAI_ENDPOINT"],
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    auth_type="API_KEY",
    auth_profile=os.getenv("OCI_PROFILE", "DEFAULT"),
    auth_file_location=os.path.expanduser(os.getenv("OCI_CONFIG_FILE", "~/.oci/config")),
    input_type="SEARCH_QUERY",
)
query_vector = query_embeddings.embed_query(query)

matches = top_matches(query_vector, document_vectors, documents, k=2)
results_df = pd.DataFrame(matches)
results_df["score"] = results_df["score"].round(4)
results_df

## Step 8 - Answer With Retrieved Context

Now we combine the two ideas:

1. Use embeddings to find relevant text.
2. Use a prompt chain to answer using only that text.


In [ ]:
# Convert retrieved rows into a compact context block for the prompt.
context = "\n\n".join(
    f"[source={row['source']}]\n{row['text']}"
    for row in matches
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context. If the answer is not in the context, say you do not know."),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}"),
])

rag_chain = rag_prompt | chat_model | StrOutputParser()
rag_answer = invoke_text_chain(
    rag_chain,
    {"context": context, "question": query},
    "the RAG answer chain",
)

sources = "\n".join(f"- `{row['source']}`" for row in matches)
display(Markdown(f"""
**Question**

> {query}

**Answer**

{rag_answer}

**Retrieved Sources**

{sources}
"""))


## When To Use LangChain vs The OCI SDK

| Use the OCI SDK when... | Use LangChain when... |
|---|---|
| You want direct control over OCI request objects | You want reusable prompt/model/parser chains |
| You are writing service-specific integration code | You are composing LLM app building blocks |
| You want fewer abstractions | You want a common interface across models and tools |
| You are debugging exact API payloads | You are building application flows quickly |

The two approaches are not competitors. LangChain still calls OCI Generative AI underneath; it just gives application developers a higher-level interface.


## Troubleshooting

| Symptom | Likely cause | What to check |
|---|---|---|
| Auth error | OCI config profile is missing or invalid | Run notebook `00`; check `OCI_PROFILE` and `OCI_CONFIG_FILE` |
| Model not found | Model is unavailable in the configured region | Check `GENAI_ENDPOINT` and `CHAT_MODEL_ID` |
| Empty Gemini output | Completion budget used for reasoning | Increase `CHAT_MAX_COMPLETION_TOKENS` |
| Embedding error | Wrong embedding model or missing permission | Check `EMBED_MODEL_ID` and IAM policy |
| Import error | Dependencies are not synced | Run `uv sync` from the files directory |
